# Fixing top-bottom qpath error

In [4]:
import os

import numpy as np
import tifffile

# The artifact
file_path = '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_5/tif/rep2_Mouse_5_bot_max_proj.tif'

print(f"Bypassing TIFF structure for: {os.path.basename(file_path)}")

with tifffile.TiffFile(file_path) as tif:
    # 1. Analyze the only valid page (Channel 0) to understand the architecture
    page0 = tif.pages[0]
    
    # Critical Check: Is the data laid out in a single flat block?
    # If this is False (i.e., tiled), this raw bypass is much riskier.
    is_contiguous = page0.is_contiguous
    is_tiled = page0.is_tiled
    
    shape = page0.shape
    dtype = page0.dtype
    
    # Get the exact byte location where Channel 0 starts and ends
    # dataoffsets is a tuple; for contiguous data, it has 1 element.
    c0_offset = page0.dataoffsets[0]
    c0_bytes = page0.databytecounts[0]
    
    print("-" * 60)
    print("Architecture Check:")
    print(f"   Contiguous: {is_contiguous}")
    print(f"   Tiled:      {is_tiled}")
    print(f"   Shape:      {shape}")
    print(f"   Dtype:      {dtype}")
    print(f"   Chan 0 Start Offset: {c0_offset} bytes")
    print(f"   Chan 0 Byte Size:    {c0_bytes / (1024**3):.2f} GB")
    print("-" * 60)

# 2. The Backdoor: Map the entire file as a raw binary blob
# We treat the file as a 1D array of bytes (uint8)
raw_file = np.memmap(file_path, dtype='uint8', mode='r')

# Calculate the hypothetical positions of the hidden channels
# We assume standard uncompressed writing: Chan 1 follows immediately after Chan 0
c1_start = c0_offset + c0_bytes
c1_end = c1_start + c0_bytes

c2_start = c1_end
c2_end = c2_start + c0_bytes

# Verify we don't read past the end of the file
file_size = os.path.getsize(file_path)
print(f"Total File Size: {file_size} bytes")
print(f"End of Chan 2 would be: {c2_end} bytes")

if c2_end > file_size:
    print("[WARNING] The file is smaller than expected for 3 contiguous channels.")
    print("We will attempt to load what exists.")

# 3. Extraction Loop
offsets = [
    ('Channel_0', c0_offset, c0_offset + c0_bytes),
    ('Channel_1', c1_start, c1_end),
    ('Channel_2', c2_start, c2_end)
]

results = {}

for name, start, end in offsets:
    if end > file_size:
        print(f"[{name}] IMPOSSIBLE: Offset implies data outside physical file.")
        continue

    print(f"[{name}] Extracting raw bytes {start}:{end}...")
    
    try:
        # Slice the raw bytes
        raw_chunk = raw_file[start:end]
        
        # Transmute bytes back into the correct image format (uint16)
        # We must 'view' the bytes as uint16, then 'reshape' to the 2D image
        image_flat = raw_chunk.view(dtype)
        image_2d = image_flat.reshape(shape)
        
        # Quick validation
        mean_val = np.mean(image_2d)
        print(f"   Success. Mean Intensity: {mean_val:.2f}")
        
        # Only keep the stats to save RAM, unless you want to process 'image_2d' now
        results[name] = mean_val
        
        # Clean up
        del image_2d
        del image_flat
        del raw_chunk
        
    except Exception as e:
        print(f"   [FAILURE] Could not reconstruct matrix: {e}")

print("-" * 60)
print("Final Status:", results)

Bypassing TIFF structure for: rep2_Mouse_5_bot_max_proj.tif
------------------------------------------------------------
Architecture Check:
   Contiguous: True
   Tiled:      False
   Shape:      (30912, 35163)
   Dtype:      uint16
   Chan 0 Start Offset: 3031 bytes
   Chan 0 Byte Size:    2.02 GB
------------------------------------------------------------
Total File Size: 6521754967 bytes
End of Chan 2 would be: 6521754967 bytes
[Channel_0] Extracting raw bytes 3031:2173920343...
   Success. Mean Intensity: 29160.47
[Channel_1] Extracting raw bytes 2173920343:4347837655...
   Success. Mean Intensity: 28088.87
[Channel_2] Extracting raw bytes 4347837655:6521754967...
   Success. Mean Intensity: 27868.57
------------------------------------------------------------
Final Status: {'Channel_0': np.float64(29160.472409895414), 'Channel_1': np.float64(28088.87208254773), 'Channel_2': np.float64(27868.570676268664)}


In [5]:
import os

import numpy as np
import tifffile

# The artifact location
file_path = '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_5/tif/rep2_Mouse_5_bot_max_proj.tif'

print("Bypassing surface layer. Targeting deep storage...")

with tifffile.TiffFile(file_path) as tif:
    # 1. Acquire the architectural blueprint from the header
    page0 = tif.pages[0]
    shape = page0.shape
    dtype = page0.dtype
    
    # Calculate the jump coordinates (size of one layer)
    # We trust the file structure is contiguous based on previous analysis
    c0_offset = page0.dataoffsets[0]
    c_bytes = page0.databytecounts[0]

# Calculate extraction vectors
c1_start = c0_offset + c_bytes
c1_end = c1_start + c_bytes
c2_start = c1_end
c2_end = c2_start + c_bytes

# 2. Map the raw binary construct
# mode='r' ensures the source remains immutable
raw_file = np.memmap(file_path, dtype='uint8', mode='r')

try:
    print(f"Extracting Channel 1 (Offset: {c1_start})...")
    # Extract bytes -> View as correct Type -> Reshape to Image
    c1_raw = raw_file[c1_start:c1_end]
    channel_1 = c1_raw.view(dtype).reshape(shape)

    print(f"Extracting Channel 2 (Offset: {c2_start})...")
    c2_raw = raw_file[c2_start:c2_end]
    channel_2 = c2_raw.view(dtype).reshape(shape)

    # 3. Materialize the 3D Volume
    # We stack them into shape (Channels, Y, X). This consumes ~4GB RAM.
    print("Materializing 3D volume in memory...")
    image_volume = np.stack([channel_1, channel_2], axis=0)
    
    # Sever the connection to the raw map to free resources
    del c1_raw, c2_raw, channel_1, channel_2
    
    print("-" * 60)
    print("[OPERATIONAL] Volume Loaded.")
    print(f"   Shape: {image_volume.shape}")
    print(f"   Dtype: {image_volume.dtype}")
    print(f"   RAM Usage: {image_volume.nbytes / (1024**3):.2f} GB")

except Exception as e:
    print("[FAILURE] The connection was severed.")
    print(f"   Error: {e}")

Bypassing surface layer. Targeting deep storage...
Extracting Channel 1 (Offset: 2173920343)...
Extracting Channel 2 (Offset: 4347837655)...
Materializing 3D volume in memory...
------------------------------------------------------------
[OPERATIONAL] Volume Loaded.
   Shape: (2, 30912, 35163)
   Dtype: uint16
   RAM Usage: 4.05 GB


In [7]:
image_volume.shape

(2, 30912, 35163)

In [8]:
import napari

In [9]:
viewer = napari.Viewer(title = 'vis corrupt tfs')
viewer.add_image(image_volume, channel_axis=0)

/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (30912, 35163) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (30912, 35163) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


[<Image layer 'Image' at 0x754b146af2d0>,
 <Image layer 'Image [1]' at 0x754b142c6390>]

In [14]:
dask_image = da.from_zarr('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_5/zarr/rep2_Mouse_5_bot_max_proj.ome.zarr/s0')

In [18]:
viewer.add_image(dask_image[1:3], channel_axis=0)

/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (30912, 35163) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (30912, 35163) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


[<Image layer 'Image [2]' at 0x754b043c6710>,
 <Image layer 'Image [3]' at 0x754b21ff5610>]

In [20]:
file_path = '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_5/tif/rep2_Mouse_5_bot_max_proj.tif'

valid_frames = []
metadata_report = {}

print("Initiating Safe-Mode Extraction...")

try:
    with tifffile.TiffFile(file_path) as tif:
        # Capture global metadata from the first valid header
        if tif.pages:
            metadata_report['shape'] = tif.pages[0].shape
            metadata_report['dtype'] = tif.pages[0].dtype
            metadata_report['compression'] = tif.pages[0].compression
            print(f"Detected Structure: {metadata_report}")

        # Iterate manually to isolate corruption
        # We use the page index to track where the break happens
        for i, page in enumerate(tif.pages):
            try:
                print(f"Materializing Frame {i} (Tiled: {page.is_tiled})...")
                
                # page.asarray() automatically stitches tiles back into a full image
                data = page.asarray()
                valid_frames.append(data)
                
            except Exception as e:
                print(f"[WARNING] Frame {i} is corrupted. Data stream severed.")
                print(f"  Reason: {e}")
                # We stop at the first sign of instability to preserve what we have
                break

except Exception as e:
    print("[CRITICAL FAILURE] The file header is completely unreadable.")
    print(f"  Error: {e}")

# Check what we salvaged
if valid_frames:
    # Stack the valid frames into a volume (Channels, Y, X)
    image_volume = np.stack(valid_frames, axis=0)
    
    print("-" * 60)
    print("[OPERATIONAL] Salvaged Volume.")
    print(f"   Shape: {image_volume.shape}")
    print(f"   Channels Recovered: {len(valid_frames)}")
    
    # # --- Visualization of Salvaged Data ---
    # custom_colors = ['#1a9641', '#a6d96a', '#978897', '#d1d1ca', '#f1b6da', '#d02c91']
    # cmap = LinearSegmentedColormap.from_list("custom_piyg", custom_colors)

    # plt.figure(figsize=(10, 10))
    # sns.set_style("white")
    # sns.despine(offset=10, trim=True)

    # # Display the first valid channel
    # ax = sns.heatmap(
    #     image_volume[0, :, :], 
    #     cmap=cmap, 
    #     cbar=True, 
    #     xticklabels=False, 
    #     yticklabels=False,
    #     square=True
    # )
    # plt.title(f"Recovered Channel 0 (Tiling Corrected)")
    # plt.show()

else:
    print("[FAILURE] No valid data could be extracted.")

Initiating Safe-Mode Extraction...
Detected Structure: {'shape': (30912, 35163), 'dtype': dtype('uint16'), 'compression': <COMPRESSION.NONE: 1>}
Materializing Frame 0 (Tiled: False)...


<TiffTag.fromfile> raised TiffFileError('<tifffile.TiffTag 555 @2226787673> invalid data type 517')
<TiffTag.fromfile> raised TiffFileError('<tifffile.TiffTag 588 @2226787685> invalid data type 591')
<TiffTag.fromfile> raised TiffFileError('<tifffile.TiffTag 634 @2226787697> invalid data type 617')
<TiffTag.fromfile> raised TiffFileError('<tifffile.TiffTag 574 @2226787709> invalid data type 574')
<TiffTag.fromfile> raised TiffFileError('<tifffile.TiffTag 579 @2226787721> invalid data type 748')
<TiffTag.fromfile> raised TiffFileError('<tifffile.TiffTag 686 @2226787733> invalid data type 584')
<TiffTag.fromfile> raised TiffFileError('<tifffile.TiffTag 640 @2226787745> invalid data type 706')
<TiffTag.fromfile> raised TiffFileError('<tifffile.TiffTag 538 @2226787757> invalid data type 587')
<TiffTag.fromfile> raised TiffFileError('<tifffile.TiffTag 419 @2226787769> invalid data type 433')
<TiffTag.fromfile> raised TiffFileError('<tifffile.TiffTag 309 @2226787781> invalid data type 367')


Materializing Frame 1 (Tiled: False)...
[CRITICAL FAILURE] The file header is completely unreadable.
  Error: corrupted tag list @7536754


ValueError: all input arrays must have the same shape